# ABNB hotels — booking-share sanity check

Two independent reads of Airbnb's hotel share, set against what management and Yipit have said.

1. **NIQ receipts (`SHARE_NIQ.SRS.HOTEL_ITEMS`).** Airbnb's hotel product writes the listing title as `<HOTEL NAME>, <ROOM TYPE>` with no redaction. An order is a hotel booking when its title starts with a hotel title from our scrape (`BOTS.ABNB.HOTELS`) followed by a comma, or equals it. Share of orders, nights and booking value by **booking month** (receipt email date). Lower bound: only hotels in the scrape count. From July 2026 the hotel titles drop out of NIQ's parse (cause unknown, raised with NIQ), so treat Jul+ as missing, not as a fall.
2. **Supra review requests (`SHARE_NIQ.SRS.SUPRA_QUALITY_MERCHANT_*`).** Share of first review-request emails ("Write a review for X" / "Write X review by …") whose host name contains a hotel word. Host names are mostly blanked, so this is keyword-only: `hotel_word` (just "hotel") is the cleaner series, `hotel_kw` (hotel|inn|motel|suites|resort|hostel) is broader and picks up host-run look-alikes. One week per month (days 1–7) to keep each query small. Measures completed stays, like Yipit's review series.

**Benchmarks**
- Airbnb Q4'25 call: hotels "a single-digit percent of total nights booked, but growing, you know, nearly double that of the overall platform."
- Q1'26: "all the top-line metrics for hotels are growing, more than double that of the entire business."
- Q2'26: "hotel nights are growing approximately three times faster than our homes business."
- Yipit (Q2 preview, 2026-07-21): hotel guest reviews +80% Y/Y through May-26, 3% of global reviews.

Both NIQ sources are the US panel, so expect levels below the global figures.

In [1]:
import pandas as pd
import snowflake.connector

pd.set_option("display.width", 200)

# Keyring CredWrite fails on this machine (WinError 1783), so don't cache the token.
conn = snowflake.connector.connect(connection_name="default", client_store_temporary_credential=False)


def q(sql: str) -> pd.DataFrame:
    df = conn.cursor().execute(sql).fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df

## 1. NIQ receipts: hotel share of orders, nights and value

In [2]:
receipts = q("""
    WITH s AS (  -- scraped hotel titles; drop short generic titles and Roami (apartment operator)
        SELECT DISTINCT UPPER(TRIM(title)) AS t FROM bots.abnb.hotels
        WHERE LENGTH(title) >= 12 AND title NOT ILIKE 'roami%'),
    ie AS (      -- order value + booking date (first receipt email)
        SELECT order_id, MAX(order_total) AS order_total, MIN(email_sent_time)::DATE AS sent
        FROM share_niq.srs.items_extended
        WHERE merchant_name IN ('AirBnB', 'AirBnB Receipts') AND order_date BETWEEN '2025-09-01' AND '2026-10-14'
        GROUP BY 1),
    hi AS (
        SELECT order_id, ANY_VALUE(UPPER(TRIM(hotel_name))) AS hn, MAX(nights) AS nights
        FROM share_niq.srs.hotel_items
        WHERE merchant_name IN ('AirBnB', 'AirBnB Receipts') AND res_start_date >= '2025-09-01'
        GROUP BY 1),
    m AS (SELECT DISTINCT hi.order_id FROM hi JOIN s ON STARTSWITH(hi.hn, s.t || ',') OR hi.hn = s.t),
    j AS (
        SELECT DATE_TRUNC('month', ie.sent) AS month, hi.nights, ie.order_total, m.order_id IS NOT NULL AS is_hotel
        FROM ie JOIN hi ON ie.order_id = hi.order_id LEFT JOIN m ON m.order_id = hi.order_id
        WHERE hi.nights BETWEEN 1 AND 60 AND ie.order_total BETWEEN 1 AND 50000)
    SELECT month, COUNT(*) AS orders, COUNT_IF(is_hotel) AS hotel_orders,
           SUM(nights) AS nights, SUM(IFF(is_hotel, nights, 0)) AS hotel_nights,
           SUM(order_total) AS value, SUM(IFF(is_hotel, order_total, 0)) AS hotel_value
    FROM j WHERE month >= '2025-10-01' GROUP BY 1 ORDER BY 1
""")

r = receipts.set_index("month")
out = pd.DataFrame({
    "orders": r.orders, "hotel_orders": r.hotel_orders,
    "pct_orders": 100 * r.hotel_orders / r.orders,
    "pct_nights": 100 * r.hotel_nights / r.nights,
    "pct_value": 100 * r.hotel_value / r.value,
    "hotel_adr": r.hotel_value / r.hotel_nights,
    "all_adr": r.value / r.nights,
    "hotel_los": r.hotel_nights / r.hotel_orders,
    "all_los": r.nights / r.orders,
}).round(2)
out

,orders,hotel_orders,pct_orders,pct_nights,pct_value,hotel_adr,all_adr,hotel_los,all_los
month,,,,,,,,,
2025-10-01,59350,1,0.00,0.00,0.00,215.88,276.17,2.00,3.71
2025-11-01,35174,0,0.00,0.00,0.00,NaN,279.50,NaN,3.84
2025-12-01,33968,0,0.00,0.00,0.00,NaN,304.53,NaN,3.85
2026-01-01,43048,13,0.03,0.02,0.01,221.52,316.61,2.08,3.89
2026-02-01,41499,54,0.13,0.06,0.05,232.97,312.67,1.89,3.89
2026-03-01,48271,130,0.27,0.15,0.10,213.60,319.36,2.09,3.77
2026-04-01,43135,103,0.24,0.15,0.11,225.23,314.36,2.37,3.70
2026-05-01,46286,134,0.29,0.17,0.12,226.20,327.52,2.16,3.69
2026-06-01,45826,104,0.23,0.14,0.09,225.15,339.03,2.21,3.67


## 2. Supra review requests: hotel share of completed stays

In [3]:
HOTEL_RE = r".*\\b(hotel|hotels|inn|motel|suites|resort|hostel)\\b.*"

rows = []
for start in pd.date_range("2024-09-01", "2026-09-01", freq="MS"):
    end = start + pd.Timedelta(days=6)
    df = q(f"""
        WITH e AS (SELECT subject_id FROM share_niq.srs.supra_quality_merchant_emails
                   WHERE email_sent_date BETWEEN '{start:%Y-%m-%d}' AND '{end:%Y-%m-%d}'),
             h AS (SELECT subject_id, subject FROM share_niq.srs.supra_quality_merchant_email_header
                   WHERE domain = 'airbnb'
                     AND (subject ILIKE 'write a review for %' OR subject ILIKE 'write % review by %'))
        SELECT COUNT(*) AS review_emails,
               COUNT_IF(REGEXP_LIKE(h.subject, '{HOTEL_RE}', 'i')) AS hotel_kw,
               COUNT_IF(h.subject ILIKE '%hotel%') AS hotel_word
        FROM e JOIN h ON e.subject_id = h.subject_id
    """)
    rows.append(df.assign(week=start))

supra = pd.concat(rows).set_index("week")
supra["pct_hotel_kw"] = (100 * supra.hotel_kw / supra.review_emails).round(2)
supra["pct_hotel_word"] = (100 * supra.hotel_word / supra.review_emails).round(2)
supra["hotel_word_yoy"] = (100 * (supra.pct_hotel_word / supra.pct_hotel_word.shift(12) - 1)).round(0)
supra

C:\Users\fintan.creedon\AppData\Local\Temp\ipykernel_59152\3475634202.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  end = start + pd.Timedelta(days=6)


C:\Users\fintan.creedon\AppData\Local\Temp\ipykernel_59152\3475634202.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  end = start + pd.Timedelta(days=6)


C:\Users\fintan.creedon\AppData\Local\Temp\ipykernel_59152\3475634202.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  end = start + pd.Timedelta(days=6)


C:\Users\fintan.creedon\AppData\Local\Temp\ipykernel_59152\3475634202.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  end = start + pd.Timedelta(days=6)


C:\Users\fintan.creedon\AppData\Local\Temp\ipykernel_59152\3475634202.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  end = start + pd.Timedelta(days=6)


...

C:\Users\fintan.creedon\AppData\Local\Temp\ipykernel_59152\3475634202.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  end = start + pd.Timedelta(days=6)


C:\Users\fintan.creedon\AppData\Local\Temp\ipykernel_59152\3475634202.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  end = start + pd.Timedelta(days=6)


C:\Users\fintan.creedon\AppData\Local\Temp\ipykernel_59152\3475634202.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  end = start + pd.Timedelta(days=6)


C:\Users\fintan.creedon\AppData\Local\Temp\ipykernel_59152\3475634202.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  end = start + pd.Timedelta(days=6)


,review_emails,hotel_kw,hotel_word,pct_hotel_kw,pct_hotel_word,hotel_word_yoy
week,,,,,,
2024-09-01,26735,66,36,0.25,0.13,NaN
2024-10-01,23732,75,40,0.32,0.17,NaN
2024-11-01,18704,51,25,0.27,0.13,NaN
2024-12-01,18098,54,19,0.30,0.10,NaN
2025-01-01,24788,116,67,0.47,0.27,NaN
2025-02-01,15217,72,31,0.47,0.20,NaN
2025-03-01,18211,59,19,0.32,0.10,NaN
2025-04-01,20186,67,28,0.33,0.14,NaN
2025-05-01,19949,86,43,0.43,0.22,NaN


## 3. Side by side

In [4]:
side = out[["pct_orders", "pct_nights", "pct_value"]].join(
    supra[["pct_hotel_word", "pct_hotel_kw"]].rename(index=lambda d: d.replace(day=1)), how="outer")
side.loc["2025-06-01":]

,pct_orders,pct_nights,pct_value,pct_hotel_word,pct_hotel_kw
2025-06-01,NaN,NaN,NaN,0.24,0.58
2025-07-01,NaN,NaN,NaN,0.24,0.63
2025-08-01,NaN,NaN,NaN,0.29,0.59
2025-09-01,NaN,NaN,NaN,0.35,0.74
2025-10-01,0.00,0.00,0.00,0.32,0.51
2025-11-01,0.00,0.00,0.00,0.54,0.91
2025-12-01,0.00,0.00,0.00,0.28,0.68
2026-01-01,0.03,0.02,0.01,0.50,0.96
2026-02-01,0.13,0.06,0.05,0.41,0.80
2026-03-01,0.27,0.15,0.10,0.42,0.76
